In [0]:
%sql
CREATE VIEW olist_catalog.olist_semantic.sales_metrics
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1

source: olist_catalog.olist_gold.fact_sales
comment: Comprehensive sales metrics for Olist e-commerce platform with customer, product, and seller analysis

joins:
  - name: customers
    source: olist_catalog.olist_gold.dim_customers
    on: source.customer_key = customers.customer_id
  - name: products
    source: olist_catalog.olist_gold.dim_products
    on: source.product_key = products.product_id
  - name: sellers
    source: olist_catalog.olist_gold.dim_sellers
    on: source.seller_key = sellers.seller_id
  - name: time_dim
    source: olist_catalog.olist_gold.dim_time
    on: source.order_date_key = time_dim.date_key

dimensions:
  - name: order_status
    expr: source.order_status
    display_name: Order Status
    comment: Current status of the order
    synonyms:
      - status
      - order state
  - name: product_category
    expr: products.product_category_name
    display_name: Product Category
    comment: Category of products sold
    synonyms:
      - category
      - product type
  - name: customer_region
    expr: customers.customer_region
    display_name: Customer Region
    comment: Geographic region of the customer
    synonyms:
      - region
      - customer location
  - name: seller_region
    expr: sellers.seller_region
    display_name: Seller Region
    comment: Geographic region of the seller
    synonyms:
      - seller location
  - name: order_year_month
    expr: time_dim.year_month
    display_name: Order Year-Month
    comment: Year and month when the order was placed
    synonyms:
      - month
      - order period
      - time period

measures:
  - name: total_revenue
    expr: SUM(source.price)
    display_name: Total Revenue
    comment: Sum of all product prices
    format:
      type: currency
      currency_code: BRL
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - revenue
      - sales
      - total sales
  - name: order_count
    expr: COUNT(DISTINCT source.order_id)
    display_name: Order Count
    comment: Number of unique orders
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - orders
      - number of orders
  - name: avg_order_value
    expr: SUM(source.price) / COUNT(DISTINCT source.order_id)
    display_name: Average Order Value
    comment: Average revenue per order
    format:
      type: currency
      currency_code: BRL
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - AOV
      - average order
  - name: total_freight
    expr: SUM(source.freight_value)
    display_name: Total Freight Cost
    comment: Sum of all shipping costs
    format:
      type: currency
      currency_code: BRL
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - freight
      - shipping cost
  - name: on_time_delivery_rate
    expr: AVG(source.on_time_delivery_flag)
    display_name: On-Time Delivery Rate
    comment: Percentage of orders delivered on or before estimated date
    format:
      type: percentage
      decimal_places:
        type: exact
        places: 1
    synonyms:
      - delivery performance
      - on time rate
$$

In [0]:
%sql
REFRESH MATERIALIZED VIEW olist_catalog.olist_semantic.sales_metrics

In [0]:
%sql
SELECT * FROM olist_catalog.olist_semantic.sales_metrics LIMIT 5